# 02 — Technical Indicators

This notebook computes common indicators used in quantitative pipelines:
- RSI (momentum)
- MACD (trend + momentum)
- Bollinger Bands (volatility envelope)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

csv_path = Path("../../data/ohlcv_sample.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
else:
    np.random.seed(0)
    dates = pd.date_range("2022-01-01", periods=700, freq="B")
    close = 100 + np.cumsum(np.random.normal(0.03, 1.0, len(dates)))
    df = pd.DataFrame({
        "Open": close + np.random.normal(0, 0.5, len(dates)),
        "High": close + np.abs(np.random.normal(0.5, 0.2, len(dates))),
        "Low": close - np.abs(np.random.normal(0.5, 0.2, len(dates))),
        "Close": close,
        "Volume": np.random.randint(900000, 5000000, len(dates)),
    }, index=dates)

print("Rows:", len(df))

## 1) Try using `ta`, fallback to manual formulas

In [ ]:
try:
    import ta
    TA_AVAILABLE = True
except Exception:
    TA_AVAILABLE = False

print("ta available:", TA_AVAILABLE)

In [ ]:
prices = df["Close"]

if TA_AVAILABLE:
    df["RSI_14"] = ta.momentum.RSIIndicator(close=prices, window=14).rsi()
    macd = ta.trend.MACD(close=prices, window_fast=12, window_slow=26, window_sign=9)
    df["MACD"] = macd.macd()
    df["MACD_SIGNAL"] = macd.macd_signal()
    bb = ta.volatility.BollingerBands(close=prices, window=20, window_dev=2)
    df["BB_MID"] = bb.bollinger_mavg()
    df["BB_HIGH"] = bb.bollinger_hband()
    df["BB_LOW"] = bb.bollinger_lband()
else:
    delta = prices.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["RSI_14"] = 100 - (100 / (1 + rs))

    ema12 = prices.ewm(span=12, adjust=False).mean()
    ema26 = prices.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_SIGNAL"] = df["MACD"].ewm(span=9, adjust=False).mean()

    rolling_mean = prices.rolling(20).mean()
    rolling_std = prices.rolling(20).std()
    df["BB_MID"] = rolling_mean
    df["BB_HIGH"] = rolling_mean + 2 * rolling_std
    df["BB_LOW"] = rolling_mean - 2 * rolling_std

df = df.dropna().copy()
print(df[["RSI_14", "MACD", "MACD_SIGNAL", "BB_HIGH", "BB_LOW"]].head())

## 2) Visualize indicators

In [ ]:
tail = df.tail(220)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(tail.index, tail["Close"], label="Close")
axes[0].plot(tail.index, tail["BB_MID"], label="BB Mid")
axes[0].plot(tail.index, tail["BB_HIGH"], label="BB High", linestyle="--")
axes[0].plot(tail.index, tail["BB_LOW"], label="BB Low", linestyle="--")
axes[0].set_title("Close + Bollinger Bands")
axes[0].legend(loc="upper left")

axes[1].plot(tail.index, tail["RSI_14"], color="tab:green")
axes[1].axhline(70, color="red", linestyle="--", alpha=0.7)
axes[1].axhline(30, color="blue", linestyle="--", alpha=0.7)
axes[1].set_title("RSI (14)")

axes[2].plot(tail.index, tail["MACD"], label="MACD")
axes[2].plot(tail.index, tail["MACD_SIGNAL"], label="Signal")
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_title("MACD")
axes[2].legend(loc="upper left")

plt.tight_layout()

## 3) Save engineered dataset

In [ ]:
feature_cols = [
    "Open", "High", "Low", "Close", "Volume",
    "RSI_14", "MACD", "MACD_SIGNAL",
    "BB_MID", "BB_HIGH", "BB_LOW"
]

out_path = Path("../../data/ohlcv_with_indicators.csv")
df[feature_cols].to_csv(out_path)
print("Saved:", out_path.resolve())

## 4) Exercises

1. Add a 20-day moving average and compare to Bollinger middle band.
2. Count days where RSI > 70 and RSI < 30.
3. Create a boolean feature `MACD_BULLISH = MACD > MACD_SIGNAL`.

Next notebook: `03_normalization_and_windows.ipynb`.